In [1]:
# use kernel anndata_to_seurat, /n/home12/abartlett04/.conda/envs/anndata_to_seurat
# uses pandas 1.5.3, matplotlib 3.6.3
import os
from pathlib import Path
from scipy import io
import pandas as pd
from scanpy import AnnData
import scanpy as sc
import numpy as np

/n/home12/abartlett04/.conda/envs/anndata_to_seurat/lib/python3.11/site-packages/anndata/utils.py:429: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/n/home12/abartlett04/.conda/envs/anndata_to_seurat/lib/python3.11/site-packages/anndata/utils.py:429: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)
/n/home12/abartlett04/.conda/envs/anndata_to_seurat/lib/python3.11/site-packages/anndata/utils.py:429: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/n/home12/abartlett04/.conda/envs/anndata_to_seurat/lib/python3.11/site-packages/anndata/utils.py:429: FutureWarning: Importing read_mtx from `anndata` is deprecated. Import anndata.io.read_mtx instead.
  warnings.warn(msg, FutureWarning)
/n/home12/abartlett04/.conda/envs/annd

In [5]:
def save_data_for_R(adata, save_dir, layer='counts', cell_metadata=None, gene_metadata=None):
    """
    Save single-cell data in Matrix Market format for use in R. Save counts data, cell metadata, and gene metadata in
    separate files.
    
    Args:
        - adata (AnnData): AnnData object containing single-cell data
        - save_dir (str): path to directory where output files will be saved
        - layer (str): name of the key in adata.layers that contains the counts data (default: 'counts')
        - cell_metadata (pd.DataFrame): DataFrame containing cell metadata (default: adata.obs)
        - gene_metadata (pd.DataFrame): DataFrame containing gene metadata (default: adata.var)
        
    Returns:
        None
    """
    
    # Check that input is valid
    if not isinstance(adata, AnnData):
        raise ValueError("Input 'adata' must be an AnnData object.")
    if layer not in adata.layers.keys():
        raise KeyError(f"Counts data not found in adata.layers. Please make sure counts data is stored under the key {layer}.")
    
    # Get default cell and gene metadata from adata.obs and adata.var, respectively
    if cell_metadata is None:
        cell_metadata = adata.obs.copy()
    if gene_metadata is None:
        gene_metadata = adata.var.copy()
    
    # Create directory if necessary
    Path(save_dir).mkdir(parents=True, exist_ok=True)
    
    # Save counts data
    if adata.layers[layer].dtype == np.float64 or adata.layers[layer].dtype == np.float32:
        adata.layers[layer].data = adata.layers[layer].data.astype(np.int16)
    io.mmwrite(os.path.join(save_dir, 'counts.mtx'), adata.layers[layer])
    
    # Save cell metadata
    cell_metadata = cell_metadata.copy()
    cell_metadata['Barcode'] = cell_metadata.index
   
    if 'X_umap' in adata.obsm.keys():
        if 'UMAP1' not in cell_metadata.columns:
            cell_metadata['UMAP1'] = adata.obsm['X_umap'][:, 0]
        if 'UMAP2' not in cell_metadata.columns:
            cell_metadata['UMAP2'] = adata.obsm['X_umap'][:, 1]
    else:
        print("adata.obsm['X_umap'] does not exist.")

    cell_metadata.to_csv(os.path.join(save_dir, 'counts_cellMeta.csv'), index=None)
    
    # Save gene metadata
    gene_metadata = gene_metadata.copy()
    gene_metadata['GeneName'] = gene_metadata.index
    gene_metadata.to_csv(os.path.join(save_dir, 'counts_geneMeta.csv'), index=None)
    
    print(f"Single-cell data successfully saved in {save_dir}")

In [2]:
sce_binder = sc.read_h5ad("../data/sce_binder.h5ad")


In [4]:
sce_binder

AnnData object with n_obs × n_vars = 282377 × 26187
    obs: 'orig.ident', 'nCount_originalexp', 'nFeature_originalexp', 'Donor', 'Age', 'AgeBin', 'Sex', 'Disease_Status', 'Classification', 'PMI', 'Hemisphere', 'Brain.pH', 'RIN', 'Smoking_Status', 'freezer_storage_time_in_days', 'lib_batch', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mito', 'log1p_total_counts_mito', 'pct_counts_mito', 'n_counts', 'n_genes', 'dd_doublet', 'dd_score', 'leiden_r1', 'leiden_r0.8', 'leiden_r0.5', 'celltype_r1', 'celltype_r0.8', 'celltypes_final', 'major_celltypes', 'ident'
    uns: 'X_name'
    obsm: 'X_PCA', 'X_UMAP'
    layers: 'logcounts'

In [7]:
save_data_for_R(sce_binder, save_dir = "../data/sce_binder_for_R/", layer = "logcounts")


adata.obsm['X_umap'] does not exist.
Single-cell data successfully saved in ../data/sce_binder_for_R/


In [4]:
sce_binder.layers["counts"] = np.expm1(sce_binder.layers["logcounts"])


In [6]:
save_data_for_R(sce_binder, save_dir = "../data/sce_binder_for_R_raw/", layer = "counts")


adata.obsm['X_umap'] does not exist.
Single-cell data successfully saved in ../data/sce_binder_for_R_raw/
